## التنبؤ بالنتيجة النهائية لحيوان في الملجأ



### خطة البحث
    - وصف مجموعة البيانات والميزات
    - تحليل البيانات الاستكشافية
    - التحليل البصري للميزات
    - الأنماط والرؤى وخصائص البيانات
    - المعالجة المسبقة للبيانات
    - التحقق من الصحة، وضبط المعلمة الفائقة
    - التحقق ومنحنيات التعلم
    - التنبؤ لعينات الصمود والاختبار
    - تقييم النموذج مع وصف المقاييس
    - الاستنتاجات



### الجزء الأول. وصف مجموعة البيانات والميزات



مجموعة البيانات من [صفحة Kaggle](https://www.kaggle.com/aaronschlegel/austin-animal-center-shelter-outcomes-and/home). تحتوي مجموعة البيانات على الميزات التالية:



#### تحتوي مجموعة البيانات من مأوى مركز أوستن للحيوانات على معلومات حول الحيوانات الموجودة في المأوى ونتائجها. ومن الضروري بناء نموذج يتنبأ بنتيجة بقاء الحيوان في الملجأ.



يمكنك رؤية الميزات أدناه:



- __age_upon_outcome__ - عمر الحيوان وقت خروجه من الملجأ.
- __معرف_الحيوان__ 
- __animal_type__ - قطة أو كلب أو أي شيء آخر (بما في ذلك خفاش واحد على الأقل!).
- __سلالة__ - سلالة الحيوانات. العديد من الحيوانات هي سلالات مختلطة عامة، على سبيل المثال. "مزيج طويل الشعر".
- __اللون__ - لون فراء الحيوان إذا كان له فراء.
- __تاريخ_الميلاد__
- __التاريخ والوقت__
- __شهر سنة__
- __الاسم__
- __نوع_النتيجة_الفرعي__
- __outcome_type__ - النتيجة النهائية لهذا الحيوان. تشمل الإدخالات المحتملة النقل، [الرحمة] القتل الرحيم، المعتمد.
- __الجنس_عند_النتيجة__


In [ ]:
import pandas as pd
import seaborn as sns
import string
import numpy as np
from sklearn.preprocessing import LabelEncoder
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import  RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [ ]:
path = 'shelter.csv'

In [ ]:
df = pd.read_csv(path, parse_dates=['date_of_birth','datetime','monthyear'])
df.head()


### الجزء الثاني. تحليل البيانات الاستكشافية



نرى مزيجًا من الميزات الفئوية والرقمية والتاريخ.
في مهمة التنبؤ بنتيجة حيوان في دار للأيتام، يكون المتغير المستهدف هو result_subtype. يحتوي result_subtype على عدة أنواع، لذلك يتم تقليل المهمة إلى تصنيف متعدد الفئات

In [ ]:
df.info()


1. تحتوي العديد من الميزات على قيم فارغة، وسنقوم بتصحيح هذا في المستقبل.
2. للتنبؤ بالنتيجة، لن نستخدم نوعًا فرعيًا للنتيجة، لذا يجب إزالته.


In [ ]:
df.drop('outcome_subtype', axis=1, inplace=True)


يمكن ملء بعض القيم الخالية، على سبيل المثال، بمتوسط أو متوسط، ولكن ليس في حالة ميزتي sex_upon_outcome و result_type، لأن هذه المتغيرات مهمة وتتكون من قيم فئوية، لذلك نتخلص أيضًا من القيم الخالية.


In [ ]:
df = df[(~df.sex_upon_outcome.isnull()) & (~df.outcome_type.isnull())]
# Empty names fill in "Unknown"
df.name = df.name.fillna('Unknown')


دعونا نرى على ميزة العمر


In [ ]:
df.age_upon_outcome.unique()


القيم لها تنسيق مختلف: بالأسابيع والأيام والسنوات. دعونا نعيد حساب العمر بناءً على تاريخ الميلاد ويوم تسجيل المعلومات في الملف ونجمع كل شيء في تنسيق واحد - اليوم. ثم ننظر إلى وصف القيم التي تم الحصول عليها.


In [ ]:
df['age_in_days'] =  (df['datetime'] - df['date_of_birth']).dt.days
df['age_in_days'].describe()

In [ ]:
#Replace negative values with zero.
df[df['age_in_days']<0]=0

In [ ]:
df.drop('age_upon_outcome', axis=1, inplace=True)
df.drop(['date_of_birth','datetime', 'monthyear'], axis=1, inplace=True)
df.drop('animal_id', axis=1, inplace=True)

In [ ]:
df.info()


الآن، لا تحتوي جميع الميزات على قيم فارغة ويمكننا المتابعة إلى مزيد من الإجراءات.


In [ ]:
df.nunique()


دعونا نحاول إجراء التحويل على ميزات السلسلة، ربما يؤدي ذلك إلى تقليل عدد القيم الفريدة


In [ ]:
def punctuation_free(text):
    text = text.replace('/', ' ')
    return ''.join([char for char in text if char not in string.punctuation])

In [ ]:
df = df[df.animal_type!=0]

In [ ]:
strings = ['animal_type','breed','color','name']
for i in strings:
    df[i] = df[i].apply(lambda x: punctuation_free(x.lower()))
df.nunique()


وفي الواقع، ساعدنا هذا التحول في تقليل عدد القيم الفريدة. يمكنك محاولة تطبيق أي تحويلات أخرى، لكننا سنتوقف عند هذا الحد ونمضي قدمًا.


In [ ]:
df['outcome_type'].value_counts()


### الجزء 3. التحليل البصري للميزات


In [ ]:
plt.figure(figsize=(12,4))
sns.countplot(y=df['outcome_type'], 
              palette='mako_r',
              order=df['outcome_type'].value_counts().index)
plt.show()


ونحن نرى أن النتيجة جيدة في معظم الحيوانات، حيث أن التبني يأخذ النصيب الأكبر، والنتائج السلبية نادرة جداً. في هذه الحالة، نرى أن الطبقات ليست متوازنة حقًا، وحقيقة أنه ليس لديهم الكثير من فرص العمل، يبدو أنه من الصحيح التنبؤ بالموت أو أن أيًا من هذه النتائج غير المحتملة ستكون مشكلة.


In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=df,
              x='animal_type',
              hue='outcome_type')
plt.legend(loc='upper right')
plt.show()

ويبدو أن توزيع النتائج يختلف أيضًا عن أنواع الحيوانات؛ يمكننا أن نرى بوضوح أن الكلاب من المرجح أن تُعاد إلى مالكها وتكون مرتبطة بها أكثر من القطط. والحيوانات من فئة أخرى أكثر عرضة للقتل الرحيم من أي نتيجة أخرى.


In [ ]:
g = sns.FacetGrid(df, hue="animal_type", size=12)
g.map(sns.kdeplot, "age_in_days") 
g.add_legend()
g.set(xlim=(0,5000), xticks=range(0,5000,365))
plt.show(g)


يمكننا أن نرى الاتجاه هنا، إذا نظرنا عن كثب، فسنرى أن هذه القمم تحدث عندما يكمل الحيوان عامًا آخر. من المنطقي أن نعتقد أن ملاجئ الحيوانات ستضع حدودًا للعمر عند تحديد ما يجب فعله مع الحيوان. على سبيل المثال، عندما يكمل الحيوان 4 سنوات، ويعاني منه. أو ربما لا يعرفون العمر الدقيق للحيوان، لذا فإن عمود "تاريخ الميلاد"، الذي حسبنا منه تواريخنا، هو مجرد تقدير تقريبي.



ويبدو أن معظم القطط يتم تبنيها خلال الأشهر الأولى، كما نرى أن هناك اتجاهاً سنوياً، وأن هناك العديد من الإنجازات في السنة الأولى. 



### الجزء 3. الأنماط والرؤى وخصائص البيانات



وبما أن لدينا بيانات فئوية، فمن الضروري تشفيرها، بما في ذلك المتغير المستهدف
بالنسبة للهدف، سنستخدم أداة تشفير الملصقات، أما بالنسبة للميزات الفئوية الأخرى فإننا نقارنها بطريقتين: 
- احصل على الدمى + LabelEncoder 
- LabelEncoder فقط



### الجزء الرابع. المعالجة المسبقة للبيانات


In [ ]:
df.head()

In [ ]:
df.sex_upon_outcome.unique()


sex_upon_outcomeدعونا نضيف ميزات جديدة للجنس بدلاً من sex_upon_outcome


In [ ]:
df['Intact'] = df.sex_upon_outcome.apply(lambda x: 1 if 'Intact' in x else 0)
df['Spayed'] = df.sex_upon_outcome.apply(lambda x: 1 if 'Spayed' in x else 0)
df['Neutered'] = df.sex_upon_outcome.apply(lambda x: 1 if 'Neutered' in x else 0)
df['Male'] = df.sex_upon_outcome.apply(lambda x: 1 if 'Male' in x else 0)
df['Female'] = df.sex_upon_outcome.apply(lambda x: 1 if 'Female' in x else 0)
df['Unknown_sex'] = df.sex_upon_outcome.apply(lambda x: 1 if 'Unknown' in x else 0)

In [ ]:
df = df.drop('sex_upon_outcome', axis = 1)
df.head()

In [ ]:
X_df = df.drop('outcome_type', axis=1)[:20000]
y_df = df['outcome_type'][:20000]


دعونا نحاول تطبيق LaberEncoder على الميزات.
لإجراء حسابات أسرع، قم بتقليل حجم العينة (لا حاجة للقيام بذلك إذا كان لديك ما يكفي من الطاقة)


In [ ]:
le = LabelEncoder()

X_df.name = le.fit_transform(X_df['name'])
X_df.animal_type = le.fit_transform(X_df['animal_type'])
X_df.color = le.fit_transform(X_df['color'])
X_df.breed = le.fit_transform(X_df['breed'])

In [ ]:
y_df = le.fit_transform(y_df)

In [ ]:
X_df.head()


### اختيار النموذج 


#### هناك العديد من النماذج المختلفة لحل مشاكل التصنيف. لهذه المهمة، يقترح تقييم أداء النماذج:
 - تصنيف الجيران
 - RandomForestClassifier
 - GradientBoostingClassifier
 
 سنقوم بتكوين المعلمات الفائقة للنموذج الذي يعطي أفضل النتائج


In [ ]:
knc = KNeighborsClassifier()
rfc = RandomForestClassifier(random_state=17)
gbc = GradientBoostingClassifier(random_state=17)

X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.3)

In [ ]:
algos = []
predictions = []
data = []
algos.append(knc)
algos.append(rfc)
algos.append(gbc)
for i in algos:
    i.fit(X_train, y_train)
    data.append({'accuracy_score': accuracy_score(i.predict(X_test), y_test)})
results = pd.DataFrame(data=data, columns=['accuracy_score'],
                       index=['KNeighborsClassifier', 'RandomForestClassifier', 
                              'GradientBoostingClassifier'])

results   


دعونا نحاول تشفير البيانات باستخدام pd.get_dummies، دعونا نرى كيف ستتغير جودة النماذج


In [ ]:
X_df = df.drop('outcome_type', axis=1)[:10000]
X_df = pd.get_dummies(X_df, columns=['animal_type','color','breed'])
X_df.name = le.fit_transform(X_df['name'])
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.3)

In [ ]:
algos = []
predictions = []
data = []
algos.append(knc)
algos.append(rfc)
algos.append(gbc)
for i in algos:
    i.fit(X_train, y_train)
    data.append({'accuracy_score': accuracy_score(i.predict(X_test), y_test)})
results = pd.DataFrame(data=data, columns=['accuracy_score'],
                       index=['KNeighborsClassifier', 'RandomForestClassifier', 
                              'GradientBoostingClassifier'])

results   


حتى مع وجود كمية صغيرة من البيانات، فقد تم تدريب النموذج لفترة أطول بكثير من الطريقة الأولى.
قد تلاحظ أن مصنف KN أصبح أكثر دقة في وضع التوقعات، ولكن مع ذلك، فإن Gradient Boosting Classifier يتعامل مع هذه المهمة بشكل أفضل في كلتا الحالتين من النماذج الأخرى. كما أن النتيجة باستخدام LabelEncoder أعلى من استخدام get_dummies.
لذا، فلنأخذ الكمية الكاملة من بيانات الاختبار باستخدام Gradient Boosting Classifier وLabelEncoder ونقوم بإعداد المقاييس الفائقة على التحقق من الصحة



### الجزء 6. التحقق من الصحة وضبط المعلمات الفائقة


In [ ]:
#we take more data
X_df = df.drop('outcome_type', axis=1)[:10000]
y_df = df['outcome_type'][:10000]
X_df.name = le.fit_transform(X_df['name'])
X_df.animal_type = le.fit_transform(X_df['animal_type'])
X_df.color = le.fit_transform(X_df['color'])
X_df.breed = le.fit_transform(X_df['breed'])
y_df = le.fit_transform(y_df)

In [ ]:
X_df.shape

In [ ]:
params_grid = {'max_features': [100, 200,348]}
model_grid = GridSearchCV(gbc,params_grid, cv=5)
model_grid.fit(X_train, y_train)
model_grid.best_params_


### الجزء السابع. التحقق من الصحة ومنحنيات التعلم


In [ ]:
from sklearn.metrics import make_scorer

# Create scorer with our accuracy-function
scorer = make_scorer(accuracy_score, greater_is_better=True)

In [ ]:
X_df = df.drop('outcome_type', axis=1)[:1000]
y_df = df['outcome_type'][:1000]
X_df = pd.get_dummies(X_df, columns=['animal_type','color','breed'])
X_df.name = le.fit_transform(X_df['name'])
y_df = le.fit_transform(y_df)
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.3)

max_depth_list = [25, 35,45]
cv_errors_list = []
train_errors_list = []
valid_errors_list = []

for max_depth in max_depth_list:
    gbc = GradientBoostingClassifier(max_depth=max_depth,random_state=17, max_features = 200)

    cv_errors = cross_val_score(estimator=gbc, 
                                  X=X_train, 
                                  y=y_train, 
                                  scoring=scorer,
                                  cv=3)  
    cv_errors_list.append(cv_errors.mean())
    
    gbc.fit(X=X_train, y=y_train)
    train_error = accuracy_score(y_train, gbc.predict(X_train))   
    train_errors_list.append(train_error)
    valid_error = accuracy_score(y_test, gbc.predict(X_test))    
    valid_errors_list.append(valid_error)
    
    print(max_depth)

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(max_depth_list,cv_errors_list)
plt.plot(max_depth_list,valid_errors_list)
plt.vlines(x=max_depth_list[np.array(cv_errors_list).argmin()], 
           ymin=0.62, ymax=0.68, 
           linestyles='dashed', colors='r')

plt.legend(['Cross validation accuracy on train', 
            'accuracy on validation set', 
            'Best Max_depth value on CV'])
plt.title("Accuracy test sets.")
plt.xlabel('Max_depth value')
plt.ylabel('accuracy value')
plt.grid()

In [ ]:
gbc = GradientBoostingClassifier(random_state=17, max_features = 200)
gbc.fit(X_train, y_train)

In [ ]:
accuracy_score(gbc.predict(X_test), y_test)


### الجزء السابع. الاستنتاجات



لقد حصلنا على نتيجة جيدة، ولكن يجب علينا إجراء إعداد أكثر تفصيلاً للمعلمات الفائقة واستخدام الكمية الكاملة من البيانات في عملية التعلم. قد يكون الحل مفيدًا للملاجئ التي تقوم بجمع مثل هذه البيانات ومحاولة التنبؤ بالنتيجة التقريبية للكشف عن الحيوانات في الملجأ. وهذا أمر مهم لأنه يسمح لهم بفهم أي العلامات تؤثر بشكل أكبر على النتيجة الإيجابية للحيوانات وأيها تعد أكثر سلبية.